# UK Atmospheric Emissions GHG Intensity Analysis

Systematic analysis of greenhouse gas and CO2 emissions intensity data across all sheets and sub-tables.

**Data structure:**
- Sheet 1: Table of contents
- Sheet 2: Notes and methodology
- Sheet 3: GHG intensity (2 sub-tables: SIC codes + detailed SIC(07) groups)
- Sheet 4: CO2 intensity (2 sub-tables: SIC codes + detailed SIC(07) groups)
- Sheet 5: CO2 intensity SDG basis (time series)

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [ ]:
# File path
file_path = '../data/provatmosphericemissionsghgintensity.xlsx'

# Load Excel file
xls = pd.ExcelFile(file_path)
print(f"Sheets available: {xls.sheet_names}")

## 2. Data Extraction Functions

Functions to systematically extract and clean each table type.

In [ ]:
def extract_metadata_sheet(sheet_name):
    """
    Extract table of contents or notes sheets.
    """
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    
    # Find header row (contains consistent column names)
    header_idx = None
    for i, row in df.iterrows():
        if row.notna().sum() >= 2 and i > 0:  # Skip first row, find row with multiple values
            header_idx = i
            break
    
    if header_idx is not None:
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=header_idx)
        df = df.dropna(how='all').reset_index(drop=True)
    
    return df


def find_table_boundaries(df, col_threshold=3):
    """
    Identify where sub-tables start in a sheet based on column density.
    Returns list of (start_row, table_name) tuples.
    """
    boundaries = []
    
    for i, row in df.iterrows():
        # Check if row contains table identifier
        row_str = str(row.iloc[0]) if pd.notna(row.iloc[0]) else ""
        
        if 'Table' in row_str and (':' in row_str or 'intensity' in row_str.lower()):
            boundaries.append((i, row_str))
    
    return boundaries


def extract_intensity_table(sheet_name, start_row, end_row=None):
    """
    Extract intensity data table with proper headers.
    Handles multi-level headers for industry codes and names.
    """
    # Read raw data
    df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    
    if end_row is None:
        end_row = len(df_raw)
    
    # Extract section
    df_section = df_raw.iloc[start_row:end_row].copy()
    
    # Find header rows (SIC code, Industry name)
    # Typically 2-3 rows after table title
    header_candidates = df_section.iloc[:5]
    
    sic_row = None
    name_row = None
    
    for i, row in header_candidates.iterrows():
        row_str = ' '.join([str(x) for x in row if pd.notna(x)]).lower()
        if 'sic' in row_str and 'code' in row_str:
            sic_row = i - start_row
        elif 'industry name' in row_str or 'industry' in row_str:
            name_row = i - start_row
            break
    
    if name_row is None:
        return None
    
    # Use industry name row as header
    df = df_section.iloc[name_row+1:].copy()
    df.columns = df_section.iloc[name_row].values
    
    # Clean: remove rows with all NaN or metadata
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    df = df.reset_index(drop=True)
    
    # Identify year column (first column typically)
    year_col = df.columns[0]
    
    # Filter to numeric years only
    df = df[pd.to_numeric(df[year_col], errors='coerce').notna()].copy()
    df[year_col] = df[year_col].astype(int)
    
    # Convert remaining columns to numeric
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df


def extract_simple_timeseries(sheet_name):
    """
    Extract simple two-column time series data.
    """
    df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    
    # Find header row
    header_idx = None
    for i, row in df_raw.iterrows():
        if 'year' in str(row.iloc[0]).lower():
            header_idx = i
            break
    
    if header_idx is None:
        return None
    
    df = df_raw.iloc[header_idx:].copy()
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    
    # Convert to numeric
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    df = df.dropna(how='all')
    
    return df

## 3. Load All Sheets and Sub-tables

In [ ]:
# Dictionary to store all extracted tables
data = {}

# 1. Table of contents
print("Loading Table of Contents...")
data['toc'] = extract_metadata_sheet('Table_of_contents')

# 2. Notes
print("Loading Notes...")
data['notes'] = extract_metadata_sheet('Notes')

# 3. GHG intensity - find sub-tables
print("\nLoading GHG intensity tables...")
df_ghg_raw = pd.read_excel(file_path, sheet_name='GHG intensity', header=None)
boundaries_ghg = find_table_boundaries(df_ghg_raw)
print(f"Found {len(boundaries_ghg)} sub-tables in GHG intensity:")
for i, (row, name) in enumerate(boundaries_ghg):
    print(f"  {i+1}. Row {row}: {name[:80]}...")

# Extract GHG sub-tables
if len(boundaries_ghg) >= 1:
    # Table 1a: SIC code level
    data['ghg_sic'] = extract_intensity_table(
        'GHG intensity', 
        start_row=0, 
        end_row=boundaries_ghg[1][0] if len(boundaries_ghg) > 1 else None
    )
    
if len(boundaries_ghg) >= 2:
    # Table 1b: Detailed SIC(07) groups
    data['ghg_detailed'] = extract_intensity_table(
        'GHG intensity',
        start_row=boundaries_ghg[1][0]
    )

# 4. CO2 intensity - find sub-tables
print("\nLoading CO2 intensity tables...")
df_co2_raw = pd.read_excel(file_path, sheet_name='CO2 intensity', header=None)
boundaries_co2 = find_table_boundaries(df_co2_raw)
print(f"Found {len(boundaries_co2)} sub-tables in CO2 intensity:")
for i, (row, name) in enumerate(boundaries_co2):
    print(f"  {i+1}. Row {row}: {name[:80]}...")

# Extract CO2 sub-tables
if len(boundaries_co2) >= 1:
    # Table 2a: SIC code level
    data['co2_sic'] = extract_intensity_table(
        'CO2 intensity',
        start_row=0,
        end_row=boundaries_co2[1][0] if len(boundaries_co2) > 1 else None
    )
    
if len(boundaries_co2) >= 2:
    # Table 2b: Detailed SIC(07) groups
    data['co2_detailed'] = extract_intensity_table(
        'CO2 intensity',
        start_row=boundaries_co2[1][0]
    )

# 5. CO2 SDG basis
print("\nLoading CO2 SDG basis...")
data['co2_sdg'] = extract_simple_timeseries('CO2 intensity SDG basis')

print("\n" + "="*60)
print("Data loading complete.")
print("="*60)

In [ ]:
# Summary of loaded data
print("\nLoaded datasets summary:")
print("-" * 60)
for key, df in data.items():
    if df is not None and not df.empty:
        print(f"{key:20s}: {df.shape[0]:4d} rows × {df.shape[1]:4d} columns")
    else:
        print(f"{key:20s}: Failed to load or empty")

## 4. Data Quality Assessment

In [ ]:
def assess_data_quality(df, name):
    """
    Assess quality metrics for a dataset.
    """
    print(f"\n{'='*60}")
    print(f"Data Quality: {name}")
    print(f"{'='*60}")
    
    if df is None or df.empty:
        print("No data available.")
        return
    
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {list(df.columns[:10])}..." if len(df.columns) > 10 else f"\nColumns: {list(df.columns)}")
    
    # Missing data
    missing = df.isnull().sum()
    missing_pct = 100 * missing / len(df)
    missing_summary = pd.DataFrame({
        'Missing': missing,
        'Pct': missing_pct
    }).query('Missing > 0').sort_values('Missing', ascending=False)
    
    if not missing_summary.empty:
        print(f"\nMissing data (top 10):")
        print(missing_summary.head(10))
    else:
        print("\nNo missing data.")
    
    # Numeric columns stats
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 1:
        print(f"\nNumeric columns: {len(numeric_cols)}")
        print(df[numeric_cols].describe().iloc[:, :5])  # First 5 columns

# Assess each dataset
for key in ['ghg_sic', 'ghg_detailed', 'co2_sic', 'co2_detailed', 'co2_sdg']:
    if key in data:
        assess_data_quality(data[key], key)

## 5. Time Series Analysis

Analyse temporal trends across all tables.

In [ ]:
def extract_time_series(df, value_cols=None, label_prefix=''):
    """
    Extract time series from intensity tables.
    Returns DataFrame with years as index.
    """
    if df is None or df.empty:
        return pd.DataFrame()
    
    # Assume first column is year
    year_col = df.columns[0]
    
    if value_cols is None:
        # Use all numeric columns
        value_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if year_col in value_cols:
            value_cols.remove(year_col)
    
    ts_df = df.set_index(year_col)[value_cols].copy()
    
    # Add prefix to column names
    if label_prefix:
        ts_df.columns = [f"{label_prefix}_{col}" for col in ts_df.columns]
    
    return ts_df


# Extract total emissions intensity over time
ts_data = {}

# Look for 'Total' columns in each dataset
for key in ['ghg_sic', 'ghg_detailed', 'co2_sic', 'co2_detailed']:
    if key in data and data[key] is not None:
        df = data[key]
        # Find column with 'Total' in name
        total_cols = [col for col in df.columns if 'total' in str(col).lower()]
        if total_cols:
            ts = extract_time_series(df, total_cols, label_prefix=key)
            ts_data[key] = ts

# CO2 SDG
if 'co2_sdg' in data and data['co2_sdg'] is not None:
    df_sdg = data['co2_sdg']
    if len(df_sdg.columns) >= 2:
        year_col = df_sdg.columns[0]
        value_col = df_sdg.columns[1]
        ts_data['co2_sdg'] = df_sdg.set_index(year_col)[[value_col]]

# Combine all time series
if ts_data:
    ts_combined = pd.concat(ts_data.values(), axis=1)
    ts_combined.columns = [col.replace('Total (Excluding consumer expenditure)', 'Total')
                           for col in ts_combined.columns]
    print("\nCombined time series data:")
    print(ts_combined.head(10))

In [ ]:
# Plot total intensity trends
if 'ts_combined' in locals() and not ts_combined.empty:
    fig, ax = plt.subplots(figsize=(14, 7))
    
    for col in ts_combined.columns:
        ts_combined[col].plot(ax=ax, marker='o', label=col, linewidth=2)
    
    ax.set_xlabel('Year', fontsize=12, fontweight='bold')
    ax.set_ylabel('Emissions Intensity\n(kt CO2e / £M)', fontsize=12, fontweight='bold')
    ax.set_title('UK Emissions Intensity Trends (All Metrics)', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate percentage change
    print("\nPercentage change (first to last year):")
    print("-" * 60)
    for col in ts_combined.columns:
        first_val = ts_combined[col].dropna().iloc[0]
        last_val = ts_combined[col].dropna().iloc[-1]
        pct_change = 100 * (last_val - first_val) / first_val
        print(f"{col:40s}: {pct_change:+7.2f}%")

## 6. Sectoral Analysis

Compare emissions intensity across industrial sectors.

In [ ]:
def analyse_sectors(df, name, top_n=10):
    """
    Analyse sectoral emissions intensity.
    """
    if df is None or df.empty:
        return
    
    print(f"\n{'='*60}")
    print(f"Sectoral Analysis: {name}")
    print(f"{'='*60}")
    
    # Get latest year
    year_col = df.columns[0]
    latest_year = df[year_col].max()
    df_latest = df[df[year_col] == latest_year].copy()
    
    # Get numeric columns (sectors)
    sector_cols = [col for col in df.columns[1:] 
                   if pd.api.types.is_numeric_dtype(df[col]) 
                   and 'total' not in str(col).lower()]
    
    # Calculate mean intensity per sector
    sector_values = df_latest[sector_cols].iloc[0]
    sector_values = sector_values.dropna().sort_values(ascending=False)
    
    print(f"\nTop {top_n} sectors by intensity ({latest_year}):")
    print("-" * 60)
    for i, (sector, value) in enumerate(sector_values.head(top_n).items(), 1):
        sector_short = sector[:50] + '...' if len(sector) > 50 else sector
        print(f"{i:2d}. {sector_short:52s}: {value:7.2f}")
    
    return sector_values.head(top_n)

# Analyse each dataset
sector_results = {}
for key in ['ghg_sic', 'ghg_detailed', 'co2_sic', 'co2_detailed']:
    if key in data:
        sector_results[key] = analyse_sectors(data[key], key, top_n=10)

In [ ]:
# Plot top sectors
if sector_results:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, (key, values) in enumerate(sector_results.items()):
        if values is not None and not values.empty:
            ax = axes[idx]
            
            # Shorten labels for plotting
            labels = [label[:30] + '...' if len(label) > 30 else label 
                     for label in values.index]
            
            ax.barh(range(len(values)), values.values, color='steelblue', alpha=0.8)
            ax.set_yticks(range(len(values)))
            ax.set_yticklabels(labels, fontsize=9)
            ax.set_xlabel('Intensity (kt CO2e / £M)', fontsize=10, fontweight='bold')
            ax.set_title(f'Top Sectors: {key}', fontsize=11, fontweight='bold')
            ax.grid(axis='x', alpha=0.3)
            ax.invert_yaxis()
    
    plt.tight_layout()
    plt.show()

## 7. Comparative Analysis: GHG vs CO2

In [ ]:
# Compare GHG and CO2 at SIC level
if 'ghg_sic' in data and 'co2_sic' in data:
    ghg_df = data['ghg_sic']
    co2_df = data['co2_sic']
    
    if ghg_df is not None and co2_df is not None:
        # Get common columns (sectors)
        common_sectors = set(ghg_df.columns[1:]).intersection(set(co2_df.columns[1:]))
        common_sectors = [col for col in common_sectors 
                         if pd.api.types.is_numeric_dtype(ghg_df[col])]
        
        # Latest year
        year_col_ghg = ghg_df.columns[0]
        year_col_co2 = co2_df.columns[0]
        
        latest_year = min(ghg_df[year_col_ghg].max(), co2_df[year_col_co2].max())
        
        ghg_latest = ghg_df[ghg_df[year_col_ghg] == latest_year][common_sectors].iloc[0]
        co2_latest = co2_df[co2_df[year_col_co2] == latest_year][common_sectors].iloc[0]
        
        # Calculate ratio
        comparison = pd.DataFrame({
            'GHG': ghg_latest,
            'CO2': co2_latest,
            'GHG/CO2 Ratio': ghg_latest / co2_latest.replace(0, np.nan)
        }).sort_values('GHG/CO2 Ratio', ascending=False)
        
        print(f"\nGHG vs CO2 Comparison ({latest_year})")
        print("=" * 80)
        print("\nSectors with highest GHG/CO2 ratio (non-CO2 GHGs):")
        print(comparison.head(10))
        
        # Plot
        fig, ax = plt.subplots(figsize=(14, 8))
        
        x = np.arange(len(comparison.head(15)))
        width = 0.35
        
        ax.bar(x - width/2, comparison.head(15)['GHG'], width, 
               label='GHG', alpha=0.8, color='coral')
        ax.bar(x + width/2, comparison.head(15)['CO2'], width, 
               label='CO2', alpha=0.8, color='skyblue')
        
        # Shorten labels
        labels = [label[:25] + '...' if len(label) > 25 else label 
                 for label in comparison.head(15).index]
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('Intensity (kt CO2e / £M)', fontsize=11, fontweight='bold')
        ax.set_title(f'GHG vs CO2 Intensity Comparison ({latest_year})', 
                    fontsize=13, fontweight='bold')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()

## 8. Cumulative Statistics

Calculate aggregate metrics across all data.

In [ ]:
# Overall summary statistics
summary_stats = {}

for key in ['ghg_sic', 'ghg_detailed', 'co2_sic', 'co2_detailed']:
    if key in data and data[key] is not None:
        df = data[key]
        numeric_cols = df.select_dtypes(include=[np.number]).columns[1:]  # Exclude year
        
        summary_stats[key] = {
            'n_sectors': len(numeric_cols),
            'n_years': df.shape[0],
            'mean_intensity': df[numeric_cols].mean().mean(),
            'median_intensity': df[numeric_cols].median().median(),
            'max_intensity': df[numeric_cols].max().max(),
            'min_intensity': df[numeric_cols].min().min(),
            'std_intensity': df[numeric_cols].std().mean()
        }

summary_df = pd.DataFrame(summary_stats).T
print("\nCumulative Summary Statistics")
print("=" * 80)
print(summary_df)

## 9. Correlation Analysis

Examine relationships between sectors and time.

In [ ]:
# Temporal correlation analysis
if 'ghg_sic' in data and data['ghg_sic'] is not None:
    df = data['ghg_sic'].copy()
    year_col = df.columns[0]
    
    # Select subset of sectors for correlation
    sector_cols = [col for col in df.columns[1:10] 
                   if pd.api.types.is_numeric_dtype(df[col])]
    
    if len(sector_cols) > 2:
        df_corr = df[sector_cols].corr()
        
        # Plot heatmap
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Shorten labels
        labels = [label[:20] + '...' if len(label) > 20 else label 
                 for label in df_corr.columns]
        
        sns.heatmap(df_corr, annot=True, fmt='.2f', cmap='coolwarm', 
                   center=0, square=True, ax=ax,
                   xticklabels=labels, yticklabels=labels,
                   cbar_kws={'label': 'Correlation'})
        
        ax.set_title('Sectoral Correlation Matrix (GHG Intensity)', 
                    fontsize=13, fontweight='bold', pad=20)
        
        plt.tight_layout()
        plt.show()

## 10. Export Processed Data

In [ ]:
# Export processed data to separate sheets
output_path = '/mnt/user-data/outputs/processed_emissions_data.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for key, df in data.items():
        if df is not None and not df.empty:
            # Limit sheet name to 31 characters (Excel limit)
            sheet_name = key[:31]
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    
    # Add summary statistics
    if summary_stats:
        summary_df.to_excel(writer, sheet_name='Summary_Stats')
    
    # Add time series
    if 'ts_combined' in locals() and not ts_combined.empty:
        ts_combined.to_excel(writer, sheet_name='Time_Series')

print(f"\nProcessed data exported to: {output_path}")

## 11. Key Findings Summary

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS SUMMARY")
print("="*80)

# 1. Overall trend
if 'ts_combined' in locals() and not ts_combined.empty:
    print("\n1. OVERALL TRENDS:")
    print("-" * 80)
    for col in ts_combined.columns:
        first_val = ts_combined[col].dropna().iloc[0]
        last_val = ts_combined[col].dropna().iloc[-1]
        first_year = ts_combined[col].dropna().index[0]
        last_year = ts_combined[col].dropna().index[-1]
        pct_change = 100 * (last_val - first_val) / first_val
        
        print(f"\n{col}:")
        print(f"  {first_year}: {first_val:.2f} kt CO2e/£M")
        print(f"  {last_year}: {last_val:.2f} kt CO2e/£M")
        print(f"  Change: {pct_change:+.1f}%")

# 2. Highest intensity sectors
if sector_results:
    print("\n2. HIGHEST INTENSITY SECTORS (Latest Year):")
    print("-" * 80)
    for key, values in sector_results.items():
        if values is not None and not values.empty:
            top_sector = values.index[0]
            top_value = values.iloc[0]
            print(f"\n{key}:")
            print(f"  {top_sector}: {top_value:.2f} kt CO2e/£M")

# 3. Data coverage
print("\n3. DATA COVERAGE:")
print("-" * 80)
for key, stats in summary_stats.items():
    print(f"\n{key}:")
    print(f"  Sectors: {stats['n_sectors']}")
    print(f"  Years: {stats['n_years']}")
    print(f"  Mean intensity: {stats['mean_intensity']:.2f} kt CO2e/£M")

print("\n" + "="*80)